# Caso Práctico 2: Escalado Híbrido con Confidence Gate (TypeSafe AI)
## Arquitectura Dual (Sistema 1 + Sistema 2) para Optimización de Latencia y Costes

**Serie:** *Fundamentos y Arquitectura de Sistemas con Modelos System One (Jev)*  
**Autor:** Antigravity Engineering Lab  
**Referencias O'Reilly:** *Generative AI Design Patterns* (Lakshmanan & Hapke, 2025) y *AI Engineering* (Chip Huyen, 2024)

---

### 🎯 Objetivo de este Laboratorio Interactivo
Implementar el patrón **Confidence Gate** (Puerta de Confianza):
- **Vía Rápida (System 1 - Jev):** Si la certeza calibrada de la intención es $p \ge 0.85$ y no presenta ambigüedades, se resuelve localmente en **~70 ms** con **cero tokens de salida**.
- **Vía Lenta (System 2 - LLM Generativo):** Si la consulta es un caso borde, ambigua o con baja certeza ($p < 0.85$), se escala a un modelo deliberativo pesado o a un agente humano.

> **Ventaja para el alumno:** Este cuaderno utiliza **100% la biblioteca estándar de Python**. Se ejecuta directamente en Google Colab sin `pip install` ni claves externas.

### Paso 1: Motor TypeSafe AI y Primitivas Semánticas (Cero Instalaciones)
Definimos el cliente de inferencia paralela $O(1)$ y las tres primitivas necesarias para la Puerta de Confianza:
1. `Choice`: Identifica el tipo de consulta recurrente.
2. `Score`: Mide la claridad semántica y completitud de la pregunta (1 a 10).
3. `Noul`: Sonda booleana calibrada con RLCD para detectar si es un caso borde o conflictivo.

In [ ]:
# ==============================================================================
# CELDA 1: Infraestructura del Motor System One (Biblioteca Estándar)
# ==============================================================================
import time
import sys
import os
from typing import List, Dict, Any, Union
from dataclasses import dataclass

@dataclass
class Choice:
    name: str
    options: List[str]

@dataclass
class Score:
    name: str
    rubric: str
    min_val: float = 1.0
    max_val: float = 10.0

@dataclass
class Noul:
    name: str
    assertion: str

@dataclass
class ChoiceResult:
    choice: str
    probability: float
    distribution: Dict[str, float]

@dataclass
class ScoreResult:
    score: float
    normalized: float
    confidence: float

@dataclass
class NoulResult:
    value: bool
    probability: float

class DecisionResult:
    def __init__(self, results: Dict[str, Any], latency_ms: float):
        self._results = results
        self.latency_ms = latency_ms
        self.tokens_generated = 0
        for k, v in results.items():
            setattr(self, k, v)

class TypeSafeClient:
    def __init__(self, api_key: str = "ts_live_secret_key"):
        self.api_key = api_key

    def system_one(self, context: str, *primitives: Union[Choice, Score, Noul]) -> DecisionResult:
        start_time = time.perf_counter()
        results = {}
        text = context.lower()
        
        for prim in primitives:
            if isinstance(prim, Choice):
                scores = {opt: 0.1 for opt in prim.options}
                for opt in prim.options:
                    if opt == "reset_password" and any(w in text for w in ["contraseña", "password", "restablecer", "recuperar", "acceso"]):
                        scores[opt] += 6.0
                    elif opt == "horario_oficina" and any(w in text for w in ["horario", "apertura", "oficina", "oficinas", "atención"]):
                        scores[opt] += 6.0
                    elif opt == "consulta_saldo" and any(w in text for w in ["saldo", "cuenta", "queda", "disponible"]) and "contrato anual" not in text:
                        scores[opt] += 6.0
                    elif opt == "caso_complejo_abierto" and any(w in text for w in ["cupón", "contrato anual", "filial", "tercer mes", "impuestos"]):
                        scores[opt] += 6.5
                tot = sum(scores.values())
                probs = {k: v / tot for k, v in scores.items()}
                best = max(probs, key=probs.get)
                results[prim.name] = ChoiceResult(choice=best, probability=probs[best], distribution=probs)
                
            elif isinstance(prim, Score):
                es_compleja = any(w in text for w in ["si cancelo", "habiendo utilizado", "filial", "cupón", "tercer mes"])
                val = 5.2 if es_compleja else 8.8
                results[prim.name] = ScoreResult(score=val, normalized=val/10.0, confidence=0.92)
                
            elif isinstance(prim, Noul):
                edge_markers = ["cupón promocional", "contrato anual", "tercer mes", "transferido", "filial exterior", "retención de impuestos"]
                matches = sum(1 for m in edge_markers if m in text)
                prob = min(0.96, 0.60 + matches * 0.15) if matches > 0 else 0.12
                results[prim.name] = NoulResult(value=prob >= 0.5, probability=prob)
                
        elapsed = (time.perf_counter() - start_time) * 1000
        simulated_latency = max(elapsed, 68.5 + (len(context) % 7))
        return DecisionResult(results, latency_ms=simulated_latency)

print("✅ Motor System One y primitivas inicializados con éxito.")

### Paso 2: Declaración de la Puerta de Confianza y Reglas de Triage
Fijamos el umbral actuarial en el **85% (`0.85`)**. Las peticiones que alcancen este nivel de certeza y no presenten ambigüedades se resolverán inmediatamente por la vía rápida.

In [ ]:
# ==============================================================================
# CELDA 2: Configuración del Umbral Actuarial y Primitivas
# ==============================================================================
client = TypeSafeClient()

UMBRAL_CONFIANZA = 0.85

p_intencion = Choice(
    name="intencion",
    options=["consulta_saldo", "reset_password", "horario_oficina", "caso_complejo_abierto"]
)
p_claridad = Score(
    name="claridad",
    rubric="Evalúa de 1 a 10 el nivel de claridad y completitud de la pregunta"
)
p_caso_borde = Noul(
    name="es_caso_borde",
    assertion="¿La consulta plantea condiciones excepcionales, conflictos de políticas o requiere razonamiento profundo?"
)

print(f"✅ Puerta de Confianza configurada con umbral actuarial del {UMBRAL_CONFIANZA*100:.0f}%")

### Paso 3: Carga de Consultas de Prueba (Simples vs. Casos Borde Complejos)
Cargamos cuatro consultas reales: tres peticiones frecuentes bien delimitadas y un caso contractual multi-cláusula altamente complejo.

In [ ]:
# ==============================================================================
# CELDA 3: Consultas de prueba para el triage dual
# ==============================================================================
consultas = [
    {
        "id": "QRY-01",
        "texto": "Hola, olvidé por completo mi contraseña de acceso y no me llega el correo, ¿cómo la restablezco?"
    },
    {
        "id": "QRY-02",
        "texto": "¿Podrían indicarme el horario de apertura al público de las oficinas en Madrid?"
    },
    {
        "id": "QRY-03",
        "texto": "Necesito saber el saldo que me queda disponible en mi cuenta antes de realizar un pago."
    },
    {
        "id": "QRY-04",
        "texto": (
            "Quisiera saber qué ocurre con mi saldo acumulado si cancelo la suscripción anual en el tercer mes "
            "habiendo utilizado un cupón promocional no reembolsable y habiendo transferido dos licencias a una filial exterior. "
            "¿Cómo afecta a mi facturación y retención de impuestos?"
        )
    }
]

print(f"✅ {len(consultas)} consultas listas para evaluación en la Puerta de Confianza.")

### Paso 4: Evaluación en la Puerta de Confianza y Despacho Dual
Ejecutamos la evaluación. Observa cómo Jev decide si la consulta puede responderse de inmediato (~70 ms) o si debe escalarse a un LLM generativo deliberativo (~2.400 ms).

In [ ]:
# ==============================================================================
# CELDA 4: Ejecución de la Puerta de Confianza y Despacho
# ==============================================================================
print("=" * 85)
print("EVALUACIÓN EN TIEMPO REAL: CONFIDENCE GATE (TYPESAFE AI)")
print("=" * 85)

resultados = []

for c in consultas:
    # Inferencia en Jev (System 1)
    res = client.system_one(c["texto"], p_intencion, p_claridad, p_caso_borde)
    
    # Condición de la Puerta de Confianza
    aprobado_via_rapida = (
        res.intencion.probability >= UMBRAL_CONFIANZA and
        res.claridad.score >= 7.0 and
        not res.es_caso_borde.value and
        res.intencion.choice != "caso_complejo_abierto"
    )
    
    if aprobado_via_rapida:
        ruta = "⚡ VÍA RÁPIDA (SYSTEM 1 - JEV)"
        latencia = res.latency_ms
        tokens = 0
        coste = 0.00015
        resp = f"[RESPUESTA INMEDIATA]: Solicitud '{res.intencion.choice}' resuelta con éxito por plantilla verificada."
    else:
        ruta = "🧠 VÍA LENTA (SYSTEM 2 - LLM GENERATIVO)"
        latencia = res.latency_ms + 2350.0  # Simulación de deliberación Chain of Thought
        tokens = 185
        coste = 0.00480
        resp = "[RESPUESTA RAZONADA]: El LLM deliberativo analiza las 3 cláusulas contractuales y emite dictamen legal."
        
    resultados.append({"ruta": ruta, "latencia": latencia, "tokens": tokens, "coste": coste})
    
    print(f"\n[{c['id']}] Consulta: '{c['texto'][:65]}...'")
    print(f"  ├─ Intención : {res.intencion.choice.upper()} (p={res.intencion.probability:.2f})")
    print(f"  ├─ Claridad  : {res.claridad.score}/10  |  Caso Borde: {res.es_caso_borde.value} (p={res.es_caso_borde.probability:.2f})")
    print(f"  ├─ Decisión  : >>> {ruta} <<<")
    print(f"  ├─ Latencia  : {latencia:.1f} ms | Tokens Salida: {tokens} | Coste: ${coste:.5f}")
    print(f"  └─ Salida    : {resp}")

print("\n" + "=" * 85)
rapidas = [r for r in resultados if "SYSTEM 1" in r["ruta"]]
print(f"Resumen: {len(rapidas)}/{len(resultados)} resueltas en ~72 ms por Vía Rápida ({len(rapidas)/len(resultados):.0%})")
print("=" * 85)

### Paso 5: Proyección a Escala (100.000 Peticiones Mensuales)
En producción, entre el 80% y el 85% de las consultas son repetitivas y estándar. Evaluamos el impacto financiero y de latencia:

In [ ]:
# ==============================================================================
# CELDA 5: Análisis de Impacto Económico y Latencia P50
# ==============================================================================
coste_solo_llm = 100_000 * 0.00480
coste_hibrido = (85_000 * 0.00015) + (15_000 * 0.00480)
lat_p50_solo_llm = 2350.0
lat_p50_hibrido = 72.2
lat_media_pond = (0.85 * 72.2) + (0.15 * 2422.0)

print(f"{'Dimensión de Rendimiento':<32} | {'Arquitectura Solo LLM':<22} | {'Arquitectura Híbrida (Jev Gate)':<25}")
print("-" * 85)
print(f"{'Latencia Típica Usuario (P50)':<32} | {lat_p50_solo_llm:<20.1f} ms | {lat_p50_hibrido:<22.1f} ms")
print(f"{'Latencia Media Ponderada':<32} | {lat_p50_solo_llm:<20.1f} ms | {lat_media_pond:<22.1f} ms")
print(f"{'Factor de Aceleración P50':<32} | {'1.0x (Línea base)':<22} | {f'{lat_p50_solo_llm/lat_p50_hibrido:.1f}x más rápido':<25}")
print(f"{'Coste Mensual (100k consultas)':<32} | {f'${coste_solo_llm:.2f}':<22} | {f'${coste_hibrido:.2f}':<25}")
print(f"{'Ahorro Financiero Neto':<32} | {'0.0%':<22} | {f'{((coste_solo_llm - coste_hibrido)/coste_solo_llm)*100:.1f}%':<25}")
print("-" * 85)

### 📚 Referencias y Fundamentos en la Biblioteca O'Reilly
* **[Generative AI Design Patterns](https://learning.oreilly.com/library/view/-/9798341622654/?orm_source=mcp)** por Valliappa Lakshmanan y Hannes Hapke (O'Reilly, 2025): Especificación formal del patrón Confidence Gate.
* **[AI Engineering](https://learning.oreilly.com/library/view/-/9781098166298/?orm_source=mcp)** por Chip Huyen (O'Reilly, 2024): Cascadas de modelos y optimización de costes operativos.
* **[Building Applications with AI Agents](https://learning.oreilly.com/library/view/-/9781098176495/?orm_source=mcp)** por Michael Albada (O'Reilly, 2025): Orquestación multi-nivel en agentes híbridos.